In [ ]:
import pandas as pd
import ast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
### Functions ###
def clean_json(json_text):
    list_names = []
    converted_list = ast.literal_eval(json_text)
    for item in converted_list:
        list_names.append(item["name"])
    return list_names

def cleaning_and_preprocessing(df):
    df.dropna(inplace=True)
    df["release_date"] = pd.to_datetime(df["release_date"], errors='coerce')
    df["release_date"] = df["release_date"].dt.strftime('%d/%m/%Y')
    df.rename(columns={"title_x": "title"}, inplace=True)
    df["genres"] = df["genres"].apply(clean_json)
    df["keywords"] = df["keywords"].apply(clean_json)
    df["cast"] = df["cast"].apply(clean_json)
    df["crew"] = df["crew"].apply(clean_json)
    df["production_companies"] = df["production_companies"].apply(clean_json)
    df["production_countries"] = df["production_countries"].apply(clean_json)
    df["spoken_languages"] = df["spoken_languages"].apply(clean_json)
    df.drop(["id", "movie_id", "title_y", "budget", "revenue", "production_countries"], axis=1, inplace=True)
    return df
    
#def normalize_columns(df, columns):
    for col in columns:
        df[col] = df[col].apply(lambda x: [str(elemento).lower().replace(" ", "") for elemento in x] if isinstance(x, list) else x)
    return df 

In [ ]:
movies = pd.read_csv("Dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("Dataset/tmdb_5000_credits.csv")
df = pd.merge(movies, credits, left_on="id", right_on="movie_id")

df = cleaning_and_preprocessing(df)   
#df = normalize_columns(df, ["genres", "production_companies", "crew", "spoken_languages", "cast"])
df.to_csv('tmdb_5000_pulito.csv', index=False)
print("Dataset salvato con successo!")

In [ ]:
### Sentence Embedding e Cosine Similarity
df["cast"] = df["cast"].apply(lambda x: " ".join(x))
df["crew"] = df["crew"].apply(lambda x: " ".join(x))
df["keywords"] = df["keywords"].apply(lambda x: " ".join(x))
df["production_companies"] = df["production_companies"].apply(lambda x: " ".join(x))
sentence_var = (df["overview"] + " " + df["cast"] + " " + df["crew"] + " " + df["keywords"] + " " + df["tagline"] +" " + df["production_companies"]) 
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_sentence = model.encode(sentence_var.tolist())
#similarity_matrix = cosine_similarity(embeddings_sentence)

In [24]:
user_query = input("Inserisci la tua query: ")
query_embedding = model.encode([user_query])
search_score = cosine_similarity(query_embedding, embeddings_sentence)[0]
results = list(enumerate(search_score))
top_10 = sorted(results, key=lambda x: x[1], reverse=True)[:10]
index = [i[0] for i in top_10]
raccomended = df["title"].iloc[index]
table = df.iloc[index][["title", "release_date", "genres", "spoken_languages", "homepage"]]

print("\nFilm Consigliati per te")
print(table.to_string(index=False))


Film Consigliati per te
           title release_date                                      genres             spoken_languages                                       homepage
      This Is It   28/10/2009                        [Music, Documentary]                    [English]                  http://www.thisisit-movie.com
        The Help   09/08/2011                                     [Drama]                    [English]               http://thehelpmovie.com/#/splash
  The Blind Side   20/11/2009                                     [Drama]                    [English]              http://www.theblindsidemovie.com/
      Shark Lake   02/10/2015                                  [Thriller]                    [English] http://www.screenmedia.net/project/shark-lake/
     Phantasm II   08/07/1988 [Action, Horror, Science Fiction, Thriller]                    [English]                            http://phantasm.com
   Dirty Grandpa   21/01/2016                                    [Comedy]  